# FaceTune — полный пайплайн обучения

Ноутбук запускает все этапы последовательно:
1. Установка зависимостей
2. Монтирование Google Drive (чекпоинты сохраняются туда)
3. Загрузка кода проекта
4. Аудит датасета
5. Обучение модели
6. Оценка на тест-сете
7. Калибровка порогов
8. Диагностика ложных срабатываний (FPR)
9. Дообучение на hard negatives
10. Финальная оценка

> **Датасет** (`Rajarshi-Roy-research/Defactify_Image_Dataset`) скачивается автоматически с HuggingFace — переносить вручную ничего не нужно.

## 0. Проверка GPU

In [9]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}  |  VRAM: {mem:.1f} GB')
else:
    print('GPU не найден — обучение будет медленным. Включи Runtime > Change runtime type > T4 GPU')

GPU: Tesla T4  |  VRAM: 15.6 GB


## 1. Установка зависимостей

In [ ]:

!pip install -q \
    torch torchvision \
    datasets \
    Pillow numpy \
    streamlit \
    pandas matplotlib

## 2. Загрузка кода и монтирование Drive

**Drive нужен только для чекпоинтов** — чтобы не потерять веса модели после закрытия сессии.
Код проекта удобнее загрузить напрямую в Colab.

**Выбери один из вариантов загрузки кода (ячейки 2a или 2b):**

In [ ]:
import os, zipfile

PROJECT_DIR = '/content/FaceTune'

# Вариант A — GitHub (раскомментируй и замени URL):
# !git clone https://github.com/ИМЯ/FaceTune {PROJECT_DIR}

# Вариант B — загрузить FaceTune.zip прямо из браузера:
if not os.path.exists(PROJECT_DIR):
    from google.colab import files
    print('Загрузи FaceTune.zip когда появится диалог...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content/')
    print(f'Готово: {PROJECT_DIR}')
else:
    print(f'Уже есть: {PROJECT_DIR}')

print(os.listdir(PROJECT_DIR))

Загрузи FaceTune.zip когда появится диалог...


Saving FaceTune.zip to FaceTune.zip
Готово: /content/FaceTune
['.gitignore', '.git', 'training', '.claude', '__pycache__', 'tests', 'requirements.txt', 'README.md', 'fp.json', 'TASKS.md', 'DONE.md', 'AGENTS.md', 'model', 'train_colab.ipynb', 'app.py', 'preprocessing']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODEL_DIR = '/content/drive/MyDrive/FaceTune_checkpoints'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f'Результаты и чекпоинты → {DRIVE_MODEL_DIR}')

Mounted at /content/drive
Результаты и чекпоинты → /content/drive/MyDrive/FaceTune_checkpoints


In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Рабочая директория — чтобы относительные пути в коде работали
os.chdir(PROJECT_DIR)
print('Рабочая директория:', os.getcwd())

Рабочая директория: /content/FaceTune


## 3. Конфигурация

Все параметры в одном месте — меняй только здесь.

In [ ]:
import json
from pathlib import Path

# ── Модель ────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'resnet50'   # 'resnet50' | 'mobilenet'

# ── Обучение ──────────────────────────────────────────────────────────────────
EPOCHS      = 10
BATCH_SIZE  = 64           # T4 16 GB тянет 64; при OOM снизь до 32
LR          = 1e-4
PATIENCE    = 3
MAX_SAMPLES = None         # None = весь датасет; 5000 — быстрый тест

# ── Дообучение на hard negatives ──────────────────────────────────────────────
HN_EPOCHS       = 5
HN_LR           = 1e-5
HN_WEIGHT       = 2.5
CHECKPOINT_BY   = 'fpr'   # 'f1' | 'fpr'

# ── Пути ──────────────────────────────────────────────────────────────────────
DRIVE = Path(DRIVE_MODEL_DIR)
WEIGHTS_PATH    = DRIVE / 'model.pth'
HN_WEIGHTS_PATH = DRIVE / 'model_hn.pth'
FP_JSON_PATH    = DRIVE / 'fp.json'
HF_CACHE_DIR    = '/content/hf_cache'

Path(PROJECT_DIR, 'model').mkdir(exist_ok=True)

# ── Утилита сохранения ────────────────────────────────────────────────────────
def save_result(name: str, data: dict) -> None:
    path = DRIVE / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    print(f'  [saved] {path}')

print('Конфигурация:')
print(f'  MODEL_NAME   = {MODEL_NAME}')
print(f'  EPOCHS       = {EPOCHS}')
print(f'  BATCH_SIZE   = {BATCH_SIZE}')
print(f'  WEIGHTS_PATH = {WEIGHTS_PATH}')

Конфигурация:
  MODEL_NAME   = resnet50
  EPOCHS       = 10
  BATCH_SIZE   = 64
  WEIGHTS_PATH = /content/drive/MyDrive/FaceTune_checkpoints/model.pth


## 4. Аудит датасета

Проверяем:
- Баланс классов (известный дисбаланс 5:1 real:ai)
- Дубликаты подписей
- Распределение по генераторам (SD, DALL-E, Midjourney)
- Утечка между сплитами

In [ ]:
from training.audit import run_pipeline_audit

audit_result = run_pipeline_audit(cache_dir=HF_CACHE_DIR)
save_result('audit', audit_result)

print(json.dumps(audit_result, indent=2, default=str))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/validation-00000-of-00002.parquet:   0%|          | 0.00/333M [00:00<?, ?B/s]

data/validation-00001-of-00002.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/448M [00:00<?, ?B/s]

...

data/test-00007-of-00008.parquet:   0%|          | 0.00/449M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/42000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/45000 [00:00<?, ? examples/s]

  [saved] /content/drive/MyDrive/FaceTune_checkpoints/audit.json
{
  "train": {
    "num_rows": 42000,
    "class_balance": {
      "total": 42000,
      "real": 7000,
      "ai_generated": 35000,
      "real_pct": 16.666666666666664,
      "ai_pct": 83.33333333333334,
      "imbalance_ratio": 5.0
    },
    "duplicates": {
      "total": 42000,
      "unique_captions": 6948,
      "duplicate_groups": 6948,
      "duplicate_rows": 35052,
      "top5_duplicates": [
        [
          "A couple of giraffe standing next to each other.",
          36
        ],
        [
          "A large jetliner sitting on top of an airport tarmac.",
          30
        ],
        [
          "A large long train on a steel track.",
          24
        ],
        [
          "A giraffe that is standing in the grass.",
          18
        ],
        [
          "A white toilet sitting next to a bathroom sink.",
          18
        ]
      ]
    },
    "generator_distribution": {
      "Real": 7000,
 

## 5. Обучение

- WeightedRandomSampler компенсирует дисбаланс классов 5:1
- Чекпоинт выбирается по **val F1** (не по accuracy)
- AMP (mixed precision) включается автоматически на GPU
- Early stopping: остановка если F1 не растёт `PATIENCE` эпох подряд

Ожидаемое время на T4: **~20–30 мин** при `MAX_SAMPLES=None`

In [ ]:
import torch
from training.train import train

train(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device_str='auto',
    weights_path=WEIGHTS_PATH,
    cache_dir=HF_CACHE_DIR,
    pretrained_backbone=True,
    patience=PATIENCE,
    model_name=MODEL_NAME,
    max_samples=MAX_SAMPLES,
)

# Сохраняем метаданные чекпоинта
ckpt = torch.load(WEIGHTS_PATH, map_location='cpu')
save_result('train_checkpoint_meta', {
    'model_name':    ckpt.get('model_name'),
    'epoch':         ckpt.get('epoch'),
    'val_accuracy':  ckpt.get('val_accuracy'),
    'val_precision': ckpt.get('val_precision'),
    'val_recall':    ckpt.get('val_recall'),
    'val_f1':        ckpt.get('val_f1'),
    'weights_path':  str(WEIGHTS_PATH),
})

/content/FaceTune/training/train.py:396: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]
/content/FaceTune/training/train.py:163: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


## 6. Оценка на validation и test

Выводит: accuracy, precision, recall, F1, confusion matrix, recall по каждому генератору.

In [ ]:
from training.evaluate import evaluate, format_confusion_matrix

CLASS_NAMES = ['real', 'ai_generated']

print('=== VALIDATION ===')
val_report = evaluate(
    split='validation',
    weights_path=WEIGHTS_PATH,
    batch_size=BATCH_SIZE,
    cache_dir=HF_CACHE_DIR,
)
m = val_report['metrics']
print(f"  acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  rec={m['recall']:.4f}  f1={m['f1']:.4f}")
print(format_confusion_matrix(val_report['confusion_matrix'], CLASS_NAMES))
save_result('eval_base_val', val_report)

print()
print('=== TEST ===')
test_report = evaluate(
    split='test',
    weights_path=WEIGHTS_PATH,
    batch_size=BATCH_SIZE,
    cache_dir=HF_CACHE_DIR,
)
m = test_report['metrics']
print(f"  acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  rec={m['recall']:.4f}  f1={m['f1']:.4f}")
print(format_confusion_matrix(test_report['confusion_matrix'], CLASS_NAMES))
save_result('eval_base_test', test_report)

print()
print('Recall по генераторам (test):')
for gen, gm in test_report['per_generator'].items():
    print(f"  {gen:<12}  recall={gm['recall']:.4f}  f1={gm['f1']:.4f}")

## 7. Калибровка порогов

Модель обучалась на данных с 83% AI-изображений, но в реальности AI — ~5% трафика.
Байесовская коррекция исправляет завышенные вероятности.

Вывод: оптимальный порог, зоны Real / Uncertain / AI.

In [ ]:
from training.calibrate import calibrate

cal_result = calibrate(
    weights_path=WEIGHTS_PATH,
    prevalence=0.05,
    cache_dir=HF_CACHE_DIR,
    split='validation',
    show_curve=False,
    min_precision=0.70,
)
save_result('calibration', cal_result)

print(f"Рекомендуемый порог AI:   {cal_result['recommended_threshold']}")
print(f"Зона Uncertain:           {cal_result['uncertain_lower']} – {cal_result['uncertain_upper']}")
best = cal_result['best_metrics']
print(f"На этом пороге:  prec={best['precision']:.3f}  rec={best['recall']:.3f}  f1={best['f1']:.3f}  coverage={best['coverage']:.1%}")

## 8. Диагностика ложных срабатываний (FPR)

Находим реальные фото, которые модель неправильно помечает как AI.

Запускаем на **validation** — чтобы test остался незатронутым для честной финальной оценки.

In [ ]:
from training.diagnose import diagnose

diag_result = diagnose(
    weights_path=WEIGHTS_PATH,
    split='validation',
    top_n=2000,
    cache_dir=HF_CACHE_DIR,
)

# fp.json нужен finetune_hard_negatives — сохраняем отдельно
with open(FP_JSON_PATH, 'w') as f:
    json.dump(diag_result, f, indent=2)

# краткая сводка без огромного списка FP-индексов
diag_summary = {k: v for k, v in diag_result.items() if not k.startswith('top_')}
save_result('diagnose', diag_summary)

print(f"FPR: {diag_result['fpr']:.1%}  ({diag_result['false_positives_total']} из {diag_result['real_total']} реальных фото)")
print(f"P(AI) распределение FP: {diag_result['prob_distribution']}")
print(f"FFT center/edge ratio:  {diag_result['fft_center_edge_ratio']}")
print(f"Топ слова в подписях:   {diag_result['top_caption_words'][:10]}")
print(f"\nfp.json → {FP_JSON_PATH}")

## 9. Дообучение на hard negatives

Hard negatives — реальные фото, которые модель уверенно путает с AI.

Дообучение добавляет их с весом `HN_WEIGHT × normal_real` — целенаправленно снижает FPR без полного переобучения.

- `--checkpoint-by fpr` сохраняет эпоху с **минимальным FPR** (а не с максимальным F1)
- Мониторинг ведётся на **test** (т.к. hard negatives из validation)

In [ ]:
from training.finetune_hard_negatives import finetune

finetune(
    fp_json=FP_JSON_PATH,
    fp_split='validation',
    weights_path=WEIGHTS_PATH,
    output_path=HN_WEIGHTS_PATH,
    epochs=HN_EPOCHS,
    lr=HN_LR,
    batch_size=BATCH_SIZE,
    hard_neg_weight=HN_WEIGHT,
    checkpoint_by=CHECKPOINT_BY,
    device_str='auto',
    cache_dir=HF_CACHE_DIR,
    patience=PATIENCE,
)

## 10. Финальная оценка — сравнение базовой модели и модели с hard negatives

In [10]:
from training.evaluate import evaluate, format_confusion_matrix

CLASS_NAMES = ['real', 'ai_generated']

def fpr_from_report(report):
    cm = report['confusion_matrix']
    real_total = sum(cm[0])
    return cm[0][1] / real_total if real_total > 0 else 0.0

print('=== Базовая модель (test) ===')
base_report = evaluate(
    split='test', weights_path=WEIGHTS_PATH,
    batch_size=BATCH_SIZE, cache_dir=HF_CACHE_DIR,
)
bm = base_report['metrics']
base_fpr = fpr_from_report(base_report)
print(f"  acc={bm['accuracy']:.4f}  f1={bm['f1']:.4f}  FPR={base_fpr:.1%}")

print()
print('=== Модель + hard negatives (test) ===')
hn_report = evaluate(
    split='test', weights_path=HN_WEIGHTS_PATH,
    batch_size=BATCH_SIZE, cache_dir=HF_CACHE_DIR,
)
hm = hn_report['metrics']
hn_fpr = fpr_from_report(hn_report)
print(f"  acc={hm['accuracy']:.4f}  f1={hm['f1']:.4f}  FPR={hn_fpr:.1%}")

print()
print('=== Confusion matrix (hard negatives) ===')
print(format_confusion_matrix(hn_report['confusion_matrix'], CLASS_NAMES))

print()
print('Recall по генераторам (hard negatives):')
for gen, gm in hn_report['per_generator'].items():
    print(f"  {gen:<12}  recall={gm['recall']:.4f}  f1={gm['f1']:.4f}")

# Сохраняем сравнительный отчёт
save_result('eval_final', {
    'base': {**base_report['metrics'], 'fpr': round(base_fpr, 4)},
    'hard_negatives': {**hn_report['metrics'], 'fpr': round(hn_fpr, 4)},
    'delta': {
        'accuracy':  round(hm['accuracy']  - bm['accuracy'],  4),
        'f1':        round(hm['f1']        - bm['f1'],        4),
        'fpr_delta': round(hn_fpr          - base_fpr,        4),
    },
    'per_generator_hn': hn_report['per_generator'],
    'confusion_matrix_hn': hn_report['confusion_matrix'],
})

=== Базовая модель (test) ===
  acc=0.8715  f1=0.9248  FPR=51.1%

=== Модель + hard negatives (test) ===
  acc=0.8182  f1=0.8843  FPR=26.0%

=== Confusion matrix (hard negatives) ===
True \ Pred        real        ai_generated 
--------------------------------------------
real               5547            1953     
ai_generated       6230           31270     

Recall по генераторам (hard negatives):
  SD2.1         recall=0.8072  f1=0.8933
  SDXL          recall=0.8841  f1=0.9385
  SD3           recall=0.6976  f1=0.8219
  DALL-E 3      recall=0.8515  f1=0.9198
  Midjourney    recall=0.9289  f1=0.9632
  [saved] /content/drive/MyDrive/FaceTune_checkpoints/eval_final.json


## 11. Скачивание чекпоинтов

Чекпоинты уже сохранены в Google Drive (`FaceTune_checkpoints/`), но если хочешь скачать прямо из Colab:

In [ ]:
from google.colab import files
import shutil

# Скачать базовую модель
files.download(str(WEIGHTS_PATH))

# Скачать модель с hard negatives (раскомментируй если нужно)
# files.download(str(HN_WEIGHTS_PATH))

In [ ]:
files.download(str(HN_WEIGHTS_PATH))